# 10 — Perspective (pin-hole) camera model

**Needed for the assignment:** $\bigstar$ YES (projection only). **Study:** 1.5 h, **Practice:** 1 h.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## Idea

A camera maps a 3D point to a pixel in two steps. **Extrinsics** ($R$, $\mathbf{t}$) move the point from world coordinates into camera coordinates. **Intrinsics** ($K$) then project it onto the image: divide by the depth $Z_c$ and convert to pixels. Going back from a pixel gives only a *ray*: the depth is lost (back-projection).

## Key formulas

## Camera Model and 3D-to-2D Projection

### 1. World Coordinates to Camera Coordinates

Transform a 3D point from world coordinates into camera coordinates:

$$
\boxed{
\mathbf{X}_c=R(\mathbf{X}-\mathbf{t})
}
$$

Where:
- $\mathbf{X}$ = 3D point in world coordinates.
- $\mathbf{X}_c$ = 3D point in camera coordinates.
- $R$ = camera rotation matrix.
- $\mathbf{t}$ = camera centre in world coordinates.

### 2. Perspective Projection

Convert the 3D camera coordinates into 2D image coordinates:

$$
u=\frac{fk_u}{Z_c}X_c+u_0
$$

$$
v=\frac{fk_v}{Z_c}Y_c+v_0
$$

Where:
- $f$ = focal length.
- $k_u,k_v$ = pixel scaling factors.
- $(u_0,v_0)$ = principal point (image centre).
- $Z_c$ = depth of the point relative to the camera.

### 3. Intrinsic Camera Matrix

The projection can be written using homogeneous coordinates:

$$
\begin{bmatrix}
u\\v\\1
\end{bmatrix}
\sim K\mathbf{X}_c
$$

Where:

$$
\boxed{
K=
\begin{bmatrix}
fk_u&0&u_0\\
0&fk_v&v_0\\
0&0&1
\end{bmatrix}
}
$$

The matrix $K$ contains the camera's **intrinsic parameters**.

### 4. Camera Projection Matrix

Combine the intrinsic and extrinsic parameters:

$$
\boxed{
P=KR[\,I\mid-\mathbf{t}\,]
}
$$

The projection matrix has dimensions $3\times4$ and 11 degrees of freedom (DoF).

A world point is projected into image coordinates using:

$$
\boxed{
\mathbf{u}\sim
P
\begin{bmatrix}
\mathbf{X}\\1
\end{bmatrix}
}
$$

Here, $\mathbf{u}=[u,v,1]^T$ is in homogeneous image coordinates.

### 5. Back-Projection (Image to 3D)

Given image coordinates and known camera depth $Z_c$, recover the 3D camera coordinates:

$$
\boxed{
\mathbf{X}_c=
Z_cK^{-1}
\begin{bmatrix}
u\\v\\1
\end{bmatrix}
}
$$

Then transform back into world coordinates:

$$
\boxed{
\mathbf{X}=R^T\mathbf{X}_c+\mathbf{t}
}
$$

This requires the depth $Z_c$. Image coordinates alone do not uniquely determine a 3D point; they define a ray from the camera centre.

## Two Conventions for Translation

There are two equivalent ways to represent camera translation.

**Lecture and Lab 4:**

$$
\mathbf{X}_c=R(\mathbf{X}-\mathbf{t})
$$

Here, $\mathbf{t}$ is the camera centre in world coordinates.

**Practice notebook:**

$$
P=K[R\mid\mathbf{t}']
$$

Here:

$$
\boxed{\mathbf{t}'=-R\mathbf{t}}
$$

Both conventions describe the same camera. The difference is whether translation represents the camera centre in world coordinates or the translation applied during world-to-camera transformation.

# Worked example (on paper)

$K$ with $fk=500$ and $(u_0,v_0)=(320,240)$; $R=I$; camera centre $\mathbf{t}=(0,0,-5)$; world point $\mathbf{X}=(1,\ 0.5,\ 5)$.

**1. Camera coordinates.** $\mathbf{X}_c=\mathbf{X}-\mathbf{t}=(1,\ 0.5,\ 10)$.

**2. Project.** $u=\frac{500}{10}\cdot1+320=370$,  $v=\frac{500}{10}\cdot0.5+240=265$.

**3. With $P$.** $P=K[I\mid-\mathbf{t}]=\begin{bmatrix}500&0&320&1600\\0&500&240&1200\\0&0&1&5\end{bmatrix}$ (the last column is $K\cdot(0,0,5)$).\
$P(1,0.5,5,1)^T=(500+1600+1600,\ 250+1200+1200,\ 5+5)=(3700,2650,10)$, so the pixel is $(370,265)$ ✓.

**4. Back-project** with known $Z_c=10$: $X_c=10\cdot\frac{370-320}{500}=1$, $Y_c=10\cdot\frac{265-240}{500}=0.5$. Then $\mathbf{X}=\mathbf{X}_c+\mathbf{t}=(1,0.5,5)$ ✓.

**5. Behind the camera:** $\mathbf{X}=(0,0,-8)$ gives $Z_c=-3<0$. Its projection formula still returns a pixel, so you must filter out points with $Z_c\le0$ yourself (Lab 4 draws them in another colour).

# The same in Python / NumPy

In [2]:
import numpy as np

# intrinsics: f*k = 500 px, principal point (320, 240)
K = np.array([[500.0, 0, 320],
              [0, 500.0, 240],
              [0, 0, 1]])

# extrinsics (lecture / Lab 4 convention): Xc = R (X - t), t = camera centre
R = np.eye(3)
t = np.array([0.0, 0.0, -5.0])

X = np.array([1.0, 0.5, 5.0])        # world point
Xc = R @ (X - t)
uvw = K @ Xc
u = uvw[:2] / uvw[2]
print("Xc =", Xc, "  K Xc =", uvw, "  -> pixel", u)

# the same with the 3x4 projection matrix P = K R [I | -t]
P = K @ R @ np.hstack([np.eye(3), -t.reshape(3, 1)])
uvw2 = P @ np.append(X, 1.0)
print("P =\n", P)
print("P X =", uvw2, " -> pixel", uvw2[:2] / uvw2[2])

# back-projection: a pixel gives a ray; with known depth Zc we get the point back
Zc = Xc[2]
Xc_back = Zc * np.linalg.inv(K) @ np.array([u[0], u[1], 1.0])
X_back = R.T @ Xc_back + t
print("back-projected (Zc = 10):", X_back)

# a camera rotated by 30 deg about Y, looking at the same point
a = np.radians(30)
Ry = np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])
Xc = Ry @ (X - t)
print("rotated camera: Zc =", round(Xc[2], 4), " pixel =", np.round((K @ Xc)[:2] / Xc[2], 2))

# a point behind the camera (Zc < 0) still gets a pixel -> must be filtered!
Xb = np.array([0.0, 0.0, -8.0])
Xcb = R @ (Xb - t)
print("behind camera: Zc =", Xcb[2], " (do not draw it)")

Xc = [ 1.   0.5 10. ]   K Xc = [3700. 2650.   10.]   -> pixel [370. 265.]
P =
 [[ 500.    0.  320. 1600.]
 [   0.  500.  240. 1200.]
 [   0.    0.    1.    5.]]
P X = [3700. 2650.   10.]  -> pixel [370. 265.]
back-projected (Zc = 10): [1.  0.5 5. ]
rotated camera: Zc = 8.1603  pixel = [679.43 270.64]
behind camera: Zc = -3.0  (do not draw it)


## Where this is used in the assignment

The PCD points are already in *camera* coordinates ($R=I$, $\mathbf{t}=\mathbf{0}$). With $K=I$ the projection is simply $u=x/z$, $v=y/z$. The solution uses exactly this to put every point back on an image grid (the **range image**). Neighbouring pixels are then neighbouring points, which gives normals and connected components (topic 14).

## Course / lab link

Lab 4: the `Camera` class (`set_extrinsics` on a sphere around the object) and `draw_points`. lec01 also covers weak-perspective and orthographic cameras: they replace $Z_c$ by a constant mean depth.

## Practice

Write your solution in the empty cell under each exercise. The **expected result** tells you what you should get, not how to get it. Grey *provided data* cells are already filled in; run them first.

### Exercise 1

Same $K$ as the example ($f k=500$, $(320,240)$), camera centre $(2,0,0)$, $R=I$. Project $(2,1,4)$ and $(0,0,4)$ by hand.

> **Expected result:** $(2,1,4)\to(320,365)$; $(0,0,4)\to(70,240)$.

In [3]:
# Exercise 1: your solution


### Exercise 2

Pixel $(420,240)$ at depth $Z_c=8$, camera of the worked example (centre $(0,0,-5)$, $R=I$): which world point is it?

> **Expected result:** $\mathbf X_c=(1.6,0,8)$, so $\mathbf X=(1.6,0,3)$.

In [4]:
# Exercise 2: your solution


### Exercise 3

Project the 8 corners of the cube $[-1,1]^3$ for a camera at $(0,0,-6)$ ($R=I$), and plot them with matplotlib.

> **Expected result:** Front face ($z=-1$, $Z_c=5$): $(220,140),(220,340),(420,140),(420,340)$. Back face ($z=1$, $Z_c=7$): $(248.57,168.57),(248.57,311.43),(391.43,168.57),(391.43,311.43)$.

In [5]:
# Exercise 3: your solution


### Exercise 4

Rotate that camera by $30^\circ$ about $y$ ($R=R_y(30^\circ)$). Project the corners with **both** conventions, $R(\mathbf X-\mathbf t)$ and $R\mathbf X+\mathbf t'$ with $\mathbf t'=-R\mathbf t$, and check that they agree.

> **Expected result:** Both agree (`np.allclose` is `True`). For example $(-1,-1,-1)\to(489.14,136.48)$ and $(1,1,1)\to(712.47,329.89)$; the $x=1$ corners go past $u=640$, i.e. out of a $640\times480$ image.

In [6]:
# Exercise 4: your solution
